# Knowledge Distillation Experiments (Colab, Drive-Cached)

This notebook includes:
- persistent model/data cache on Drive
- teacher checkpoint flow
- multi-student baseline vs distillation
- ablations (embedding/logit/classification loss mixes)
- comparison plots and embedding visualizations

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/PSI_main')
REPO_URL = 'https://github.com/huz-n/knowledge_distillation.git'
REPO_DIR = DRIVE_ROOT / 'knowledge_distillation'

os.environ['TORCH_HOME'] = str(DRIVE_ROOT / 'cache' / 'torch')
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'cache' / 'hf')
os.environ['XDG_CACHE_HOME'] = str(DRIVE_ROOT / 'cache' / 'xdg')
os.environ['PIP_CACHE_DIR'] = str(DRIVE_ROOT / 'cache' / 'pip')

for p in [DRIVE_ROOT, Path(os.environ['TORCH_HOME']), Path(os.environ['HF_HOME']), Path(os.environ['XDG_CACHE_HOME']), Path(os.environ['PIP_CACHE_DIR'])]:
    p.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git pull
else:
    %cd {DRIVE_ROOT}
    !git clone {REPO_URL}
    %cd {REPO_DIR}

DATA_DIR = DRIVE_ROOT / 'data'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_DIR)
print('Data dir:', DATA_DIR)
print('Output root:', OUTPUT_ROOT)
print('TORCH_HOME:', os.environ['TORCH_HOME'])

In [ ]:
!python -m pip install -q --upgrade pip
!pip install -q -r requirements.txt
!python -c "import torch, os; print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available()); print('TORCH_HOME=', os.environ.get('TORCH_HOME'))"

In [ ]:
# One-time pretrained weight warmup (cached in TORCH_HOME on Drive)
!python -m src.warmup_model_cache --models resnet50 resnet18 mobilenetv3_small

In [ ]:
# Shared settings
DATASET = 'cifar100'
EPOCHS_TEACHER = 15
EPOCHS_STUDENT = 20
BATCH_SIZE = 96
IMAGE_SIZE = 64
AUGMENT = 'basic'
DISTILL_DIM = 256
TEACHER_PCA_SAMPLES = 10000
PROJECTOR_DEPTH = 4
PROJECTOR_HIDDEN = 512
PROJECTOR_DROPOUT = 0.1
MAX_TRAIN_BATCHES = 0  # set e.g. 120 for quick runs
MAX_VAL_BATCHES = 0


In [ ]:
TEACHER_OUT = OUTPUT_ROOT / 'teacher_resnet50_cifar100'
B_R18_OUT = OUTPUT_ROOT / 'baseline_resnet18_cifar100'
D_R18_OUT = OUTPUT_ROOT / 'distill_resnet18_bottleneck_cifar100'
B_TINYCNN_OUT = OUTPUT_ROOT / 'baseline_tinycnn_cifar100'
D_TINYCNN_OUT = OUTPUT_ROOT / 'distill_tinycnn_bottleneck_cifar100'
CMP_STUDENTS_OUT = OUTPUT_ROOT / 'compare_students_cifar100'
CMP_VARIANTS_OUT = OUTPUT_ROOT / 'compare_variants_resnet18_cifar100'
VIZ_R18_OUT = OUTPUT_ROOT / 'embedding_viz_resnet18_cifar100'

In [ ]:
# 1) Teacher checkpoint flow (train once, reuse)
if (TEACHER_OUT / 'teacher_best.pt').exists():
    print('Teacher checkpoint already exists, skipping training.')
else:
    !python -m src.train_teacher \
      --dataset {DATASET} \
      --data-dir {DATA_DIR} \
      --teacher-model resnet50 \
      --teacher-pretrained \
      --finetune-mode full \
      --augment {AUGMENT} \
      --epochs $EPOCHS_TEACHER \
      --batch-size $BATCH_SIZE \
      --image-size $IMAGE_SIZE \
      --channels-last \
      --max-train-batches $MAX_TRAIN_BATCHES \
      --max-val-batches $MAX_VAL_BATCHES \
      --output-dir {TEACHER_OUT}


In [ ]:
# 2) Core: student A (ResNet-18) baseline
!python -m src.train_baseline \
  --dataset {DATASET} \
  --data-dir {DATA_DIR} \
  --student resnet18 \
  --augment {AUGMENT} \
  --epochs $EPOCHS_STUDENT \
  --batch-size $BATCH_SIZE \
  --image-size $IMAGE_SIZE \
  --channels-last \
  --max-train-batches $MAX_TRAIN_BATCHES \
  --max-val-batches $MAX_VAL_BATCHES \
  --output-dir {B_R18_OUT}


In [ ]:
# 3) Core: student A (ResNet-18) bottleneck embedding distillation + classification
!python -m src.train_distill \
  --dataset {DATASET} \
  --data-dir {DATA_DIR} \
  --student resnet18 \
  --teacher resnet50 \
  --teacher-checkpoint {TEACHER_OUT / 'teacher_best.pt'} \
  --augment {AUGMENT} \
  --distill-dim $DISTILL_DIM \
  --teacher-pca-samples $TEACHER_PCA_SAMPLES \
  --projector-depth $PROJECTOR_DEPTH \
  --projector-hidden-dim $PROJECTOR_HIDDEN \
  --projector-dropout $PROJECTOR_DROPOUT \
  --embed-loss cosine \
  --embed-normalize \
  --cls-weight 1.0 \
  --embed-weight 0.2 \
  --logit-weight 0.0 \
  --epochs $EPOCHS_STUDENT \
  --batch-size $BATCH_SIZE \
  --image-size $IMAGE_SIZE \
  --channels-last \
  --max-train-batches $MAX_TRAIN_BATCHES \
  --max-val-batches $MAX_VAL_BATCHES \
  --output-dir {D_R18_OUT}


In [ ]:
# 4) Core: student B (tiny_cnn) baseline
!python -m src.train_baseline \
  --dataset {DATASET} \
  --data-dir {DATA_DIR} \
  --student tiny_cnn \
  --augment {AUGMENT} \
  --epochs $EPOCHS_STUDENT \
  --batch-size $BATCH_SIZE \
  --image-size $IMAGE_SIZE \
  --channels-last \
  --max-train-batches $MAX_TRAIN_BATCHES \
  --max-val-batches $MAX_VAL_BATCHES \
  --output-dir {B_TINYCNN_OUT}


In [ ]:
# 5) Core: student B (tiny_cnn) bottleneck embedding distillation + classification
!python -m src.train_distill \
  --dataset {DATASET} \
  --data-dir {DATA_DIR} \
  --student tiny_cnn \
  --teacher resnet50 \
  --teacher-checkpoint {TEACHER_OUT / 'teacher_best.pt'} \
  --augment {AUGMENT} \
  --distill-dim $DISTILL_DIM \
  --teacher-pca-samples $TEACHER_PCA_SAMPLES \
  --projector-depth $PROJECTOR_DEPTH \
  --projector-hidden-dim $PROJECTOR_HIDDEN \
  --projector-dropout $PROJECTOR_DROPOUT \
  --embed-loss cosine \
  --embed-normalize \
  --cls-weight 1.0 \
  --embed-weight 0.3 \
  --logit-weight 0.0 \
  --epochs $EPOCHS_STUDENT \
  --batch-size $BATCH_SIZE \
  --image-size $IMAGE_SIZE \
  --channels-last \
  --max-train-batches $MAX_TRAIN_BATCHES \
  --max-val-batches $MAX_VAL_BATCHES \
  --output-dir {D_TINYCNN_OUT}


In [ ]:
# 6) Student comparison (model + parameter-count views, ACC + F1)
!python -m src.compare_students \
  --run "resnet18|{B_R18_OUT / 'metrics.json'}|{D_R18_OUT / 'metrics.json'}" \
  --run "tiny_cnn|{B_TINYCNN_OUT / 'metrics.json'}|{D_TINYCNN_OUT / 'metrics.json'}" \
  --output-dir {CMP_STUDENTS_OUT}


In [ ]:
# 7) Ablations on ResNet-18 (extra experiments)
D_R18_EMBED_ONLY = OUTPUT_ROOT / 'distill_resnet18_embed_only_cifar100'
D_R18_LOGIT_CLS = OUTPUT_ROOT / 'distill_resnet18_logit_cls_cifar100'
D_R18_EMBED_LOGIT_CLS = OUTPUT_ROOT / 'distill_resnet18_embed_logit_cls_cifar100'
!python -m src.train_distill \
  --dataset {DATASET} \
  --data-dir {DATA_DIR} \
  --student resnet18 \
  --teacher resnet50 \
  --teacher-checkpoint {TEACHER_OUT / 'teacher_best.pt'} \
  --augment {AUGMENT} \
  --distill-dim $DISTILL_DIM \
  --teacher-pca-samples $TEACHER_PCA_SAMPLES \
  --projector-depth $PROJECTOR_DEPTH \
  --projector-hidden-dim $PROJECTOR_HIDDEN \
  --projector-dropout $PROJECTOR_DROPOUT \
  --embed-loss cosine \
  --embed-normalize \
  --cls-weight 0.0 \
  --embed-weight 1.0 \
  --logit-weight 0.0 \
  --epochs $EPOCHS_STUDENT \
  --batch-size $BATCH_SIZE \
  --image-size $IMAGE_SIZE \
  --channels-last \
  --max-train-batches $MAX_TRAIN_BATCHES \
  --max-val-batches $MAX_VAL_BATCHES \
  --output-dir {D_R18_EMBED_ONLY}
!python -m src.train_distill \
  --dataset {DATASET} \
  --data-dir {DATA_DIR} \
  --student resnet18 \
  --teacher resnet50 \
  --teacher-checkpoint {TEACHER_OUT / 'teacher_best.pt'} \
  --augment {AUGMENT} \
  --cls-weight 1.0 \
  --embed-weight 0.0 \
  --logit-weight 1.0 \
  --temperature 4.0 \
  --epochs $EPOCHS_STUDENT \
  --batch-size $BATCH_SIZE \
  --image-size $IMAGE_SIZE \
  --channels-last \
  --max-train-batches $MAX_TRAIN_BATCHES \
  --max-val-batches $MAX_VAL_BATCHES \
  --output-dir {D_R18_LOGIT_CLS}
!python -m src.train_distill \
  --dataset {DATASET} \
  --data-dir {DATA_DIR} \
  --student resnet18 \
  --teacher resnet50 \
  --teacher-checkpoint {TEACHER_OUT / 'teacher_best.pt'} \
  --augment {AUGMENT} \
  --distill-dim $DISTILL_DIM \
  --teacher-pca-samples $TEACHER_PCA_SAMPLES \
  --projector-depth $PROJECTOR_DEPTH \
  --projector-hidden-dim $PROJECTOR_HIDDEN \
  --projector-dropout $PROJECTOR_DROPOUT \
  --embed-loss cosine \
  --embed-normalize \
  --cls-weight 1.0 \
  --embed-weight 0.2 \
  --logit-weight 1.0 \
  --temperature 4.0 \
  --epochs $EPOCHS_STUDENT \
  --batch-size $BATCH_SIZE \
  --image-size $IMAGE_SIZE \
  --channels-last \
  --max-train-batches $MAX_TRAIN_BATCHES \
  --max-val-batches $MAX_VAL_BATCHES \
  --output-dir {D_R18_EMBED_LOGIT_CLS}


In [ ]:
# 8) Variant table/plot for report
!python -m src.compare_variants \
  --run "baseline_cls|{B_R18_OUT / 'metrics.json'}" \
  --run "embed_plus_cls|{D_R18_OUT / 'metrics.json'}" \
  --run "embed_only|{D_R18_EMBED_ONLY / 'metrics.json'}" \
  --run "logit_plus_cls|{D_R18_LOGIT_CLS / 'metrics.json'}" \
  --run "embed_logit_cls|{D_R18_EMBED_LOGIT_CLS / 'metrics.json'}" \
  --title 'ResNet-18 Distillation Ablations (CIFAR-100, PCA bottleneck)' \
  --output-dir {CMP_VARIANTS_OUT}


In [ ]:
# 9) Embedding visualization for baseline vs bottleneck distillation
!python -m src.visualize_embeddings \
  --dataset {DATASET} \
  --data-dir {DATA_DIR} \
  --student-model resnet18 \
  --teacher-model resnet50 \
  --teacher-checkpoint {TEACHER_OUT / 'teacher_best.pt'} \
  --baseline-checkpoint {B_R18_OUT / 'student_best.pt'} \
  --distill-checkpoint {D_R18_OUT / 'student_best.pt'} \
  --image-size $IMAGE_SIZE \
  --align-mode lstsq \
  --normalize-mode l2_teacher_zscore \
  --max-items 1500 \
  --output-dir {VIZ_R18_OUT}


In [ ]:
import json
from IPython.display import Image, display

print('=== Student comparison ===')
print((CMP_STUDENTS_OUT / 'summary.csv').read_text())
print('=== Variant comparison ===')
print((CMP_VARIANTS_OUT / 'summary.csv').read_text())

display(Image(filename=str(CMP_STUDENTS_OUT / 'acc_by_model.png')))
display(Image(filename=str(CMP_STUDENTS_OUT / 'acc_by_params.png')))
display(Image(filename=str(CMP_VARIANTS_OUT / 'variant_scores.png')))
display(Image(filename=str(VIZ_R18_OUT / 'pca_models.png')))
display(Image(filename=str(VIZ_R18_OUT / 'tsne_models.png')))